# Laya V2.9.1 — ALL Global FastWindowMemory, safeguarded fine-tuning and self-contained Hugging Face export

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V29_AllGlobal_FastWindowMemory_Colab.ipynb)

## Motivation and scientific guardrails

Previously measured **single-layer (Layer 15)** 400-step audit: **W1024 cosine=0.99972, NMSE=0.00057, teacher decision agreement=0.9974, speed=0.164×**; **W512 cosine=0.99490, NMSE=0.01088, agreement=0.9777, speed=0.255×**. Those results **do not** establish all-layer quality. The notebook freshly trains separate W1024/W512 modules from each layer's original QKV, Wo and RoPE weights, installs every global-attention replacement while preserving all sliding layers, records cumulative decisions, and runs optional 60-step joint decision recovery, teacher/converted comparisons plus an untouched official test split and CUDA latency.

**Default FULL_WINDOW=1024** is the most accurate measured Layer-15 candidate; at MAX_LEN=512 it is a **dense quadratic full-attention control** (not an efficient replacement). Change FULL_WINDOW=512 to test the partially local candidate. The model still computes landmark and global-linear routes. Large windows use a compact banded/dense SDPA path rather than W×(2W−1) padded halos.

Each global layer is distilled independently on original *teacher* hidden states; the student is evaluated after each installed layer to detect **compounding hidden-state distribution shift**. If a layer misses 0.99 cosine, it is still included, with a warning, to measure failure honestly. Checkpoints are saved per layer before end-to-end evaluation. No 0.99 all-layer or speed claim is implied. Run in a fresh GPU-enabled Colab runtime.



In [ ]:
#@title 1. Install
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"


In [ ]:
#@title 2. Model and V2.8 experiment configuration
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from transformers.models.modernbert.modeling_modernbert import apply_rotary_pos_emb
from tinycenn_lm.laya_lab.pdelta import PDelta3GDN2CLVRAttention
from tinycenn_lm.laya_lab.core import _depthwise_sequence_conv, _valid_tokens
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence, collate_items,
    normalize_question,
)
from tinycenn_lm.decision_training import require_finite, decision_distill_loss, guarded_step

SEED = 2028
SOURCE_MODEL = "convaiinnovations/laya-typed-decisions" #@param {type:"string"}
LAYER = 15  # reassigned to the current global layer during conversion
FULL_WINDOW = 1024 #@param {type:"integer"}
WINDOWS = (FULL_WINDOW,) #@param {type:"raw"}
LANDMARK_STRIDE = 16 #@param {type:"integer"}
FEATURE_DIM = 96 #@param {type:"integer"}
TRAIN_CASES = 600 #@param {type:"integer"}
DEV_CASES = 160 #@param {type:"integer"}
MAX_LEN = 512 #@param {type:"integer"}
BATCH = 3 #@param {type:"integer"}
STEPS = 400 #@param {type:"integer"}
RUN_JOINT_RECOVERY = True #@param {type:"boolean"}
JOINT_STEPS = 160 #@param {type:"integer"}
FINE_TUNE_TASK_HEAD = True #@param {type:"boolean"}
SUPERVISED_WEIGHT = 0.15 #@param {type:"number"}
TEACHER_WEIGHT = 0.5 #@param {type:"number"}
MAX_TEACHER_AGREEMENT_DROP = 0.01
HF_REPO_ID = "" #@param {type:"string"}
UPLOAD_TO_HF = True #@param {type:"boolean"}
HF_PRIVATE = True #@param {type:"boolean"}
VERIFY_STANDALONE_EXPORT = True #@param {type:"boolean"}
JOINT_DEV_LIMIT = 160
JOINT_BATCH = 1
TRAIN_CACHE_BATCHES = 32 #@param {type:"integer"}
PROBE_CACHE_BATCHES = 8 #@param {type:"integer"}
AUDIT_CACHE_BATCHES = 8 #@param {type:"integer"}
CHECK_EVERY = 50
TARGET_COSINE = 0.99
OUTPUT_DIR = Path("/content/Laya_V29_AllGlobal_FastWindowMemory")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert STEPS == 400, "V2.8 comparison is defined at exactly 400 optimizer steps"
assert len(WINDOWS) > 0 and all(isinstance(w, int) and w >= 16 for w in WINDOWS)
assert LANDMARK_STRIDE > 0 and FEATURE_DIM > 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a CUDA GPU (Colab > Runtime > Change runtime type).")
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda")
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

source_dir = Path(snapshot_download(
    SOURCE_MODEL,
    allow_patterns=["rl_agent_config.json", "model.safetensors", "encoder/*", "tokenizer/*"],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
student_eval, _ = build_checkpoint_model(source_dir)
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))
teacher.to(device).eval().requires_grad_(False)
student_eval.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile = False
student_eval.encoder.config.reference_compile = False

print("GPU:", torch.cuda.get_device_name(0), "| AMP:", amp_dtype)
from tinycenn_lm.standalone_decision import full_attention_indices
GLOBAL_LAYERS = full_attention_indices(teacher)
SLIDING_LAYERS = [i for i in range(len(teacher.encoder.layers)) if i not in GLOBAL_LAYERS]
assert GLOBAL_LAYERS and 15 in GLOBAL_LAYERS
for i in GLOBAL_LAYERS:
    assert tuple(teacher.encoder.layers[i].attn.local_attention) == (-1, -1)
print("Global layers:", GLOBAL_LAYERS, "| native sliding:", SLIDING_LAYERS)
print("Window:", FULL_WINDOW, "| MAX_LEN:", MAX_LEN, "| per-layer updates:", STEPS)
print("W>=2*MAX_LEN-1 gives dense full-context local attention:", FULL_WINDOW >= 2*MAX_LEN-1)
print("Teacher:", SOURCE_MODEL, "| official test split stays untouched")


In [ ]:
#@title 3. Leak-free train/dev data and teacher I/O helpers
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
dev_items = make_items_from_rows(dev_rows, SEED + 1, True)

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        got["x"] = args[0].detach()
        got["pos"] = None if kwargs.get("position_embeddings") is None else tuple(
            z.detach() for z in kwargs["position_embeddings"]
        )
        got["mask"] = None if kwargs.get("attention_mask") is None else kwargs["attention_mask"].detach()
        # ModernBERT SDPA requires position_ids and sliding_window_mask.
        # Replay the exact teacher call, including all positional/mask kwargs.
        got["call_kwargs"] = {
            key: (value.detach() if torch.is_tensor(value) else value)
            for key, value in kwargs.items()
        }
        if "position_ids" not in got["call_kwargs"] or "sliding_window_mask" not in got["call_kwargs"]:
            raise RuntimeError("Teacher attention hook did not capture ModernBERT SDPA kwargs")
    def post(mod, args, kwargs, out):
        got["y"] = out[0].detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "call_kwargs": got["call_kwargs"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred, _ = repl(entry["x"], **entry["call_kwargs"])
            require_finite(pred, "V2.8 local probe")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.10 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/dev decisions:", len(train_items), len(dev_items))
assert len(dev_items) >= BATCH * (PROBE_CACHE_BATCHES + AUDIT_CACHE_BATCHES), "Increase DEV_CASES for disjoint probe and audit caches"
print("official test remains untouched; probe/audit I/O non-overlapping")


In [ ]:
#@title 4. V2.8: bounded LocalW + landmarks + global linear memory; no quadratic global attention
class PDeltaV28FastWindowMemory(PDelta3GDN2CLVRAttention):
    """PDelta3-inspired replacement without sequential GDN scan.

    Complexity: local attention O(T*W*d), landmark cross attention
    O(T*ceil(T/stride)*d), and kernel memory O(T*F*d).
    The landmark branch becomes quadratic if stride is fixed while T -> infinity.
    For provably bounded linear scaling, cap its count with adaptive pooling.
    """
    architecture = "pdelta_v28_fast_window_memory"

    def __init__(self, original, *, local_window=96, landmark_stride=16,
                 feature_dim=96, max_landmarks=32):
        super().__init__(
            original, feature_dim=feature_dim, conv_kernel=4,
            chunk_size=32, local_kernel=5,
            local_window=local_window, local_gate_init=0.90,
        )
        self.landmark_stride = int(landmark_stride)
        self.max_landmarks = int(max_landmarks)
        assert self.landmark_stride > 0 and self.max_landmarks > 0
        # Remove obsolete recurrent, causal-convolution and four-way fusion parameters.
        # The inherited methods _local_attention and _global_linear remain in use.
        for name in (
            "wq", "wk", "decay_w", "dt_bias", "erase_w", "erase_b",
            "write_w", "write_b", "route_proj", "route_gate_w",
            "route_gate_b", "q_conv", "k_conv", "v_conv",
            "mix_logits", "mix_gate_w", "local_gate_w",
            "local_gate_b", "log_gain",
        ):
            delattr(self, name)
        # 4.57.6 passes position_ids to ModernBERT, not position_embeddings.
        # Copy the exact pretrained per-layer RoPE instead of silently skipping it.
        self.rotary_emb = copy.deepcopy(original.rotary_emb)
        self.rotary_emb.requires_grad_(False)
        self.layer_idx = int(getattr(original, "layer_id", -1))
        h, d = self.num_heads, self.head_dim
        # [local, compressed landmarks, global linear, direct value].
        # Layer 15 is global: do not initialize with ~90% exclusively local routing.
        self.route_logits = nn.Parameter(
            torch.tensor([1.0, 0.2, 0.2, -2.2]).repeat(h, 1)
        )
        self.route_query = nn.Parameter(torch.zeros(h, 4, d))
        self.local_log_temperature = nn.Parameter(torch.zeros(h))
        self.landmark_log_temperature = nn.Parameter(torch.zeros(h))
        self.conv_gain = nn.Parameter(torch.full((h,), -4.0))
        # Pretrained QKV and Wo are copied from the teacher by BaseLayaReplacementAttention.
        # Freeze QKV and allow a very small Wo calibration.
        for p in self.Wqkv.parameters():
            p.requires_grad_(False)
        for p in self.Wo.parameters():
            p.requires_grad_(True)

    def _local_attention(self, q, k, v, valid):
        """Centered local window; avoid huge empty W×(2W-1) halos when W >= T."""
        b, h, t, d = q.shape
        w = self.local_window
        if w < t:
            return super()._local_attention(q, k, v, valid)
        allowed = valid[:, None, None, :]
        if w < 2*t-1:
            qi = torch.arange(t, device=q.device)[:, None]
            ki = torch.arange(t, device=q.device)[None, :]
            band = ((ki >= qi-(w-1)//2) & (ki <= qi+w//2))
            allowed = allowed & band[None, None]
        z = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, dropout_p=0.0)
        return z * valid[:, None, :, None].to(z.dtype)

    def _landmarks(self, q, k, v, valid):
        b, h, t, d = k.shape
        # Choose at most max_landmarks pooled key/value blocks. This ensures
        # O(T*M) with M <= max_landmarks at all sequence lengths.
        stride = max(self.landmark_stride, math.ceil(t / self.max_landmarks))
        blocks = math.ceil(t / stride)
        pad = blocks * stride - t
        mask = valid[:, None, :, None].to(k.dtype)
        packed_valid = F.pad(valid, (0, pad), value=False).reshape(b, blocks, stride)
        counts = packed_valid.sum(-1)
        divisor = counts.clamp_min(1).to(k.dtype)[:, None, :, None]
        kp = F.pad(k * mask, (0, 0, 0, pad)).reshape(b, h, blocks, stride, d)
        vp = F.pad(v * mask, (0, 0, 0, pad)).reshape(b, h, blocks, stride, d)
        pooled_k = kp.sum(3) / divisor
        pooled_v = vp.sum(3) / divisor
        allowed = (counts > 0)[:, None, None, :]
        temperature = self.landmark_log_temperature.clamp(-0.4, 0.4).exp()
        q_scaled = q * temperature[None, :, None, None]
        # Only T x M attention scores; never a dense T x T matrix.
        return F.scaled_dot_product_attention(
            q_scaled, pooled_k, pooled_v, attn_mask=allowed, dropout_p=0.0
        ) * mask

    def forward(self, hidden_states, position_embeddings=None,
                attention_mask=None, position_ids=None, sliding_window_mask=None,
                **kwargs):
        # Exact teacher QKV + RoPE path; apply rotary embeddings BEFORE attention.
        b, t, _ = hidden_states.shape
        qkv = self.Wqkv(hidden_states).reshape(b, t, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.transpose(3, 1).unbind(dim=2)
        if position_embeddings is None:
            cos, sin = self.rotary_emb(qkv, position_ids=position_ids)
        else:
            cos, sin = position_embeddings
        q, k = apply_rotary_pos_emb(q, k, cos, sin)
        valid = _valid_tokens(attention_mask, hidden_states)
        mask = valid[:, None, :, None].to(v.dtype)
        q, k, v = q * mask, k * mask, v * mask
        temperature = self.local_log_temperature.clamp(-0.4, 0.4).exp()
        qlocal = q * temperature[None, :, None, None]
        local = self._local_attention(qlocal, k, v, valid)
        landmarks = self._landmarks(q, k, v, valid)
        global_linear = self._global_linear(q, k, v, mask) * mask
        qnorm = F.normalize(q, dim=-1)
        dynamic = torch.einsum("bhtd,hkd->bhtk", qnorm, self.route_query)
        mixture = torch.softmax(
            self.route_logits.to(q.dtype)[None, :, None, :] + dynamic,
            dim=-1
        )
        fused = (
            mixture[..., 0:1] * local
            + mixture[..., 1:2] * landmarks
            + mixture[..., 2:3] * global_linear
            + mixture[..., 3:4] * v
        )
        local_conv = _depthwise_sequence_conv(v, self.local_weight, causal=False)
        conv_weight = torch.sigmoid(self.conv_gain).to(v.dtype)[None, :, None, None]
        fused = (fused + conv_weight * (local_conv - v)) * mask
        return self.finish(fused, hidden_states)

    def config_dict(self):
        return dict(
            architecture=self.architecture, layer_idx=self.layer_idx,
            local_window=self.local_window, landmark_stride=self.landmark_stride,
            max_landmarks=self.max_landmarks, feature_dim=self.feature_dim,
            global_linear_memory=True, local_windowed_attention=True,
            dense_global_full_attention=self.local_window >= 2 * MAX_LEN - 1, recurrent_scan=False,
            rope_applied=True, upstream_sdpa_kwargs=True,
        )

def build_v28(window):
    return PDeltaV28FastWindowMemory(
        teacher.encoder.layers[LAYER].attn,
        local_window=int(window),
        landmark_stride=LANDMARK_STRIDE,
        max_landmarks=32,
        feature_dim=FEATURE_DIM,
    ).to(device)

def state_cpu(module):
    return {k: p.detach().cpu().clone() for k, p in module.state_dict().items()}

print("V2.8 architecture ready (no GDN scan; bounded local + 32 capped landmarks)")


In [ ]:
#@title 6. Fast 400-step teacher-I/O distillation and checkpoint selection
def score_local(m):
    return m["nmse"] + 1.50 * (1.0 - m["cosine"])

def optimizer_v28(model):
    core, output = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (output if name.startswith("Wo.") else core).append(p)
    params = core + output
    opt = torch.optim.AdamW([
        {"params": core, "lr": 0.003, "base_lr": 0.003},
        {"params": output, "lr": 0.00008, "base_lr": 0.00008},
    ], betas=(0.9, 0.95), weight_decay=2e-4)
    return opt, params

def train_400(model, train_cache, probe_cache, window):
    optimizer, params = optimizer_v28(model)
    scaler = torch.amp.GradScaler("cuda", enabled=amp_dtype == torch.float16, init_scale=128.0)
    best_metrics = local_metrics_from_cache(model, probe_cache)
    best_score = score_local(best_metrics)
    best_step = 0
    best_state = state_cpu(model)
    history = [{"step": 0, **best_metrics}]
    successes, attempts = 0, 0
    print(f"W{window} step    0/{STEPS}: NMSE={best_metrics['nmse']:.5f} cosine={best_metrics['cosine']:.5f}")

    while successes < STEPS:
        if attempts > STEPS + 300:
            raise RuntimeError("Too many AMP overflow/skipped optimizer steps")
        # A moderate cosine decay + short warmup is more stable than PDelta's
        # former 1e-2 opening learning rate, especially with trainable temperature.
        warm = min(1.0, (successes + 1) / 15.0)
        decay = 0.15 + 0.85 * 0.5 * (1.0 + math.cos(math.pi * successes / STEPS))
        for group in optimizer.param_groups:
            group["lr"] = group["base_lr"] * warm * decay

        model.train()
        model.out_drop.eval()
        entry = train_cache[attempts % len(train_cache)]
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            output, _ = model(entry["x"], **entry["call_kwargs"])
            m = entry["valid"][:, :, None].float()
            target = entry["y"].float()
            pred = output.float()
            mse = (((pred - target) ** 2) * m).sum() / (m.sum() * pred.shape[-1]).clamp_min(1)
            power = ((target ** 2) * m).sum() / (m.sum() * pred.shape[-1]).clamp_min(1)
            nmse = mse / power.clamp_min(1e-8)
            cos = F.cosine_similarity(pred[entry["valid"]], target[entry["valid"]], dim=-1).mean()
            loss = nmse + 1.50 * (1.0 - cos)
        require_finite(loss, f"V28 W{window} training loss")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        old_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        attempts += 1
        if scaler.is_enabled() and scaler.get_scale() < old_scale:
            continue
        successes += 1
        if successes == 1 or successes % CHECK_EVERY == 0 or successes == STEPS:
            probe = local_metrics_from_cache(model, probe_cache)
            history.append({"step": successes, **probe})
            s = score_local(probe)
            if s < best_score:
                best_score, best_metrics, best_step = s, probe, successes
                best_state = state_cpu(model)
            print(
                f"W{window} step {successes:4d}/{STEPS}: "
                f"NMSE={probe['nmse']:.5f} cosine={probe['cosine']:.5f} "
                f"| best cosine={best_metrics['cosine']:.5f} @ {best_step}"
            )
    model.load_state_dict(best_state, strict=True)
    model.eval()
    return {
        "best_step": best_step, "best_probe": best_metrics,
        "attempts": attempts, "successful_updates": successes,
        "history": history,
    }


In [ ]:
#@title 7. End-to-end quality and CUDA latency diagnostics
@torch.no_grad()
def validate_pair(student, items=dev_items, limit=None):
    teacher.eval()
    student.eval()
    limit = len(items) if limit is None else min(int(limit), len(items))
    total = agree = teacher_gold = student_gold = 0
    js = []
    for off in range(0, limit, BATCH):
        chosen = items[off:min(off + BATCH, limit)]
        b = direct_batch(chosen)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            tl, _ = teacher(**b)
            sl, _ = student(**b)
        require_finite(tl, "teacher validation logits")
        require_finite(sl, "V28 validation logits")
        allowed = b["marker_mask"].bool()
        tp = torch.softmax(tl.float().masked_fill(~allowed, -1e9), -1)
        sp = torch.softmax(sl.float().masked_fill(~allowed, -1e9), -1)
        gt = torch.tensor([row["gold_index"] for row in chosen], device=device)
        agree += int((tp.argmax(-1) == sp.argmax(-1)).sum())
        teacher_gold += int((tp.argmax(-1) == gt).sum())
        student_gold += int((sp.argmax(-1) == gt).sum())
        mid = 0.5 * (tp + sp)
        q = 0.5 * (tp * (tp.clamp_min(1e-8).log() - mid.clamp_min(1e-8).log())).sum(-1)
        q += 0.5 * (sp * (sp.clamp_min(1e-8).log() - mid.clamp_min(1e-8).log())).sum(-1)
        js.extend(q.cpu().tolist())
        total += len(chosen)
    return dict(
        decisions=total, agreement=agree / max(total, 1),
        mean_js=float(np.mean(js)),
        teacher_gold_accuracy=teacher_gold / max(total, 1),
        student_gold_accuracy=student_gold / max(total, 1),
    )

@torch.inference_mode()
def timed_attention(module, entry, warmup=5, repeats=15):
    module.eval()
    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        for _ in range(warmup):
            module(entry["x"], **entry["call_kwargs"])
        torch.cuda.synchronize()
        ms = []
        for _ in range(repeats):
            start = time.perf_counter()
            module(entry["x"], **entry["call_kwargs"])
            torch.cuda.synchronize()
            ms.append((time.perf_counter() - start) * 1000)
    return float(np.median(ms))

def attention_coverage(cache, window):
    lengths = torch.cat([item["valid"].sum(-1).cpu() for item in cache]).numpy()
    # A centered W=96 attends only 47 tokens left and 48 right.
    # Length <= W alone does NOT guarantee all-to-all attention.
    full_context_limit = min((window - 1) // 2, window // 2) + 1
    return {
        "sequences": int(len(lengths)),
        "median_valid_tokens": float(np.median(lengths)),
        "max_valid_tokens": int(np.max(lengths)),
        "fraction_fully_covered_by_local_window": float(np.mean(lengths <= full_context_limit)),
        "full_context_length_limit": int(full_context_limit),
        "sequences_longer_than_window": int(np.sum(lengths > window)),
        "sequences_at_least_2x_window": int(np.sum(lengths >= 2 * window)),
    }



In [ ]:
#@title 8. Convert all global attention, checkpoint each layer, audit cumulative errors
DEV_QUALITY_ITEMS = dev_items[(PROBE_CACHE_BATCHES + AUDIT_CACHE_BATCHES) * BATCH:]
assert DEV_QUALITY_ITEMS, "Need dev samples disjoint from probe and audit"
LAYER_RESULTS = []
CUMULATIVE_RESULTS = []
CHECKPOINT_DIR = OUTPUT_DIR / f"per_layer_W{FULL_WINDOW}"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
before = validate_pair(student_eval, items=DEV_QUALITY_ITEMS, limit=120)
print("BASELINE teacher vs identical student:", before)
for i in GLOBAL_LAYERS:
    LAYER = i  # build_v28() copies THIS teacher layer, NOT the Layer15 weights
    print("\n" + "="*92)
    print(f"ALL-GLOBAL DISTILLATION — ORIGINAL LAYER {i}, W{FULL_WINDOW}")
    print("="*92)
    trained_path = CHECKPOINT_DIR / f"layer{i:02d}_W{FULL_WINDOW}.pt"
    model = build_v28(FULL_WINDOW)
    if trained_path.exists():
        saved = torch.load(trained_path, map_location="cpu", weights_only=True)
        assert saved["layer"] == i and saved["window"] == FULL_WINDOW
        model.load_state_dict(saved["state_dict"], strict=True)
        result = saved["train"]
        audit = saved["audit"]
        print("RESUMED:", trained_path)
    else:
        train_cache = cache_teacher_io(i, train_items, TRAIN_CACHE_BATCHES, offset=i*23)
        probe_cache = cache_teacher_io(i, dev_items, PROBE_CACHE_BATCHES, offset=0)
        audit_cache = cache_teacher_io(
            i, dev_items, AUDIT_CACHE_BATCHES, offset=PROBE_CACHE_BATCHES*BATCH)
        # ModernBERT originally returns ONE tensor in a one-element tuple.
        with torch.no_grad(), torch.autocast("cuda", dtype=amp_dtype):
            replay = teacher.encoder.layers[i].attn(
                train_cache[0]["x"], **train_cache[0]["call_kwargs"]
            )[0]
        torch.testing.assert_close(
            replay.float(), train_cache[0]["y"].float(), atol=1e-3, rtol=1e-3)
        result = train_400(model, train_cache, probe_cache, FULL_WINDOW)
        audit = local_metrics_from_cache(model, audit_cache)
        torch.save({
            "version": "2.9", "source_model": SOURCE_MODEL,
            "window": FULL_WINDOW, "layer": i,
            "architecture": model.config_dict(), "train": result,
            "audit": audit, "state_dict": state_cpu(model),
        }, trained_path)
        del train_cache, probe_cache, audit_cache, replay
    model.eval().requires_grad_(False)
    student_eval.encoder.layers[i].attn = model
    metrics = validate_pair(student_eval, items=DEV_QUALITY_ITEMS, limit=120)
    LAYER_RESULTS.append({"layer": i, "probe": result["best_probe"], "audit": audit,
                          "best_step": result["best_step"],
                          "checkpoint": str(trained_path)})
    CUMULATIVE_RESULTS.append({"through_layer": i, "e2e": metrics})
    if audit["cosine"] < 0.99:
        print(f"WARNING: Layer {i} audit cosine {audit['cosine']:.5f} below target; retained for failure analysis.")
    print(f"Layer {i} audit cosine={audit['cosine']:.5f} NMSE={audit['nmse']:.5f}")
    print("CUMULATIVE E2E:", metrics)
    torch.cuda.empty_cache()

assert all(isinstance(student_eval.encoder.layers[i].attn, PDeltaV28FastWindowMemory)
           for i in GLOBAL_LAYERS)
assert all(not isinstance(student_eval.encoder.layers[i].attn, PDeltaV28FastWindowMemory)
           for i in SLIDING_LAYERS)
print("ALL GLOBAL LAYERS REPLACED:", GLOBAL_LAYERS)


In [ ]:
#@title 9. Fine-tune ALL replaced layers + optional task head, select best by held-out DEV
# Stage A is complete: all full attention layers already converted and distilled.
# Stage B trains them JOINTLY on supervised gold labels plus teacher consistency.
# No official test examples are used. Restore step-0 if fine-tuning hurts.
def snapshot_replacements():
    return {i: state_cpu(student_eval.encoder.layers[i].attn) for i in GLOBAL_LAYERS}

def restore_replacements(snapshot):
    for i in GLOBAL_LAYERS:
        student_eval.encoder.layers[i].attn.load_state_dict(snapshot[i], strict=True)

def selected_head_state():
    return {name: state_cpu(getattr(student_eval, name))
            for name in ("head", "type_emb", "scorer", "act_head")
            if getattr(student_eval, name) is not None}

def restore_head_state(snapshot):
    for name, state in snapshot.items():
        getattr(student_eval, name).load_state_dict(state, strict=True)

def rank_dev(m):
    # Reward actual held-out accuracy, while protecting teacher agreement.
    return (0.20 * (1 - m["agreement"])
            + 0.80 * (1 - m["student_gold_accuracy"])
            + 1.0 * m["mean_js"])

before_joint = validate_pair(student_eval, DEV_QUALITY_ITEMS, limit=JOINT_DEV_LIMIT)
JOINT_REPORT = {"enabled": RUN_JOINT_RECOVERY, "before": before_joint,
                "selected": before_joint, "best_step": 0, "successful_updates": 0}
if RUN_JOINT_RECOVERY and JOINT_STEPS > 0:
    core, wo, head = [], [], []
    for i in GLOBAL_LAYERS:
        layer = student_eval.encoder.layers[i].attn
        for name, p in layer.named_parameters():
            if name.startswith(("Wqkv.", "rotary_emb.")):
                continue
            p.requires_grad_(True)
            if name.startswith("Wo."):
                wo.append(p)
            else:
                core.append(p)
    if FINE_TUNE_TASK_HEAD:
        for name in ("head", "type_emb", "scorer", "act_head"):
            mod = getattr(student_eval, name)
            if mod is not None:
                for p in mod.parameters():
                    p.requires_grad_(True)
                    head.append(p)
    groups = [
        {"params": core, "lr": 3e-5, "weight_decay": 1e-4},
        {"params": wo, "lr": 3e-6, "weight_decay": 1e-4},
    ]
    if head:
        groups.append({"params": head, "lr": 2e-6, "weight_decay": 1e-4})
    optimizer = torch.optim.AdamW(groups, betas=(0.9, 0.95))
    scaler = torch.amp.GradScaler("cuda", enabled=amp_dtype == torch.float16,
                                  init_scale=128.0)
    best_replacements = snapshot_replacements()
    best_head = selected_head_state() if head else {}
    best_score = rank_dev(before_joint)
    best_step, updates, attempts = 0, 0, 0
    while updates < JOINT_STEPS:
        if attempts > JOINT_STEPS + 150:
            raise RuntimeError("Too many skipped AMP updates in joint fine-tuning")
        optimizer.zero_grad(set_to_none=True)
        selected_rows = [
            train_items[(9000 + attempts * JOINT_BATCH + j) % len(train_items)]
            for j in range(JOINT_BATCH)
        ]
        b = direct_batch(selected_rows)
        gold = torch.tensor([row["gold_index"] for row in selected_rows],
                            device=device, dtype=torch.long)
        with torch.no_grad(), torch.autocast("cuda", dtype=amp_dtype):
            tlog, tact = teacher(**b)
        student_eval.eval()  # dropout off, autograd still enabled
        with torch.autocast("cuda", dtype=amp_dtype):
            slog, sact = student_eval(**b)
            supervised = F.cross_entropy(slog.float(), gold)
            teacher_loss = decision_distill_loss(
                slog, tlog, b["marker_mask"].bool(), sact, tact
            )
            loss = SUPERVISED_WEIGHT * supervised + TEACHER_WEIGHT * teacher_loss
        updated = guarded_step(
            loss, optimizer, scaler,
            [(core, 0.5), (wo, 0.2)] + ([(head, 0.1)] if head else []),
            context=f"V2.9.1 joint fine-tune {updates + 1}"
        )
        attempts += 1
        if not updated:
            continue
        updates += 1
        if updates == 1 or updates % 20 == 0 or updates == JOINT_STEPS:
            candidate = validate_pair(student_eval, DEV_QUALITY_ITEMS, limit=JOINT_DEV_LIMIT)
            # Never accept a worse teacher-agreement or lower gold-accuracy
            # checkpoint just because the training loss decreased.
            agreement_floor = before_joint["agreement"] - MAX_TEACHER_AGREEMENT_DROP
            accuracy_floor = before_joint["student_gold_accuracy"]
            acceptable = (
                candidate["agreement"] >= agreement_floor
                and candidate["student_gold_accuracy"] >= accuracy_floor
            )
            score = rank_dev(candidate)
            if acceptable and score < best_score:
                best_replacements = snapshot_replacements()
                if head:
                    best_head = selected_head_state()
                best_score, best_step = score, updates
            print(f"fine-tune {updates}/{JOINT_STEPS}: "
                  f"loss={float(loss.detach()):.4f}, "
                  f"gold={candidate['student_gold_accuracy']:.4f}, "
                  f"agreement={candidate['agreement']:.4f}, "
                  f"JS={candidate['mean_js']:.6f}, "
                  f"accepted={acceptable and best_step == updates}, best={best_step}")
    restore_replacements(best_replacements)
    if head:
        restore_head_state(best_head)
    student_eval.eval().requires_grad_(False)
    selected_metric = validate_pair(student_eval, DEV_QUALITY_ITEMS, limit=JOINT_DEV_LIMIT)
    JOINT_REPORT = {
        "enabled": True, "before": before_joint, "selected": selected_metric,
        "best_step": best_step, "successful_updates": updates,
        "dev_selection_improved": best_step > 0,
        "supervised_weight": SUPERVISED_WEIGHT,
        "teacher_weight": TEACHER_WEIGHT,
        "fine_tune_head": FINE_TUNE_TASK_HEAD,
    }
print("JOINT FINE-TUNING:", JOINT_REPORT)

# Preserve all fitted parameters before any expensive official-test evaluation.
pretest_path = OUTPUT_DIR / f"v29_pretest_W{FULL_WINDOW}.pt"
torch.save({
    "version": "2.9", "window": FULL_WINDOW,
    "global_layers": GLOBAL_LAYERS, "joint_report": JOINT_REPORT,
    "replacement_states": snapshot_replacements(),
    "local_results": LAYER_RESULTS,
}, pretest_path)
print("Pre-test checkpoint saved:", pretest_path)


In [ ]:
#@title 10. Full-model evaluation and latency on held-out DEV and official TEST
dev_final = validate_pair(student_eval, items=DEV_QUALITY_ITEMS)
test_rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="test"))
test_items = make_items_from_rows(test_rows, SEED + 99, False)
test_final = validate_pair(student_eval, items=test_items)
@torch.no_grad()
def model_latency(model, selected, warmup=5, repeats=18):
    b = direct_batch(selected)
    with torch.autocast("cuda", dtype=amp_dtype):
        for _ in range(warmup):
            model(**b)
        torch.cuda.synchronize()
        values = []
        for _ in range(repeats):
            start = time.perf_counter()
            model(**b)
            torch.cuda.synchronize()
            values.append((time.perf_counter()-start)*1000)
    return {"median_ms": float(np.median(values)),
            "p95_ms": float(np.percentile(values,95)),
            "seq_len": int(b["input_ids"].shape[1])}
latency = []
for selected in [DEV_QUALITY_ITEMS[:1], DEV_QUALITY_ITEMS[:BATCH]]:
    t = model_latency(teacher, selected)
    s = model_latency(student_eval, selected)
    latency.append({"batch_size": len(selected), "teacher": t, "student": s,
                    "speedup": t["median_ms"]/max(s["median_ms"], 1e-9)})
report = {
    "version": "2.9", "window": FULL_WINDOW, "max_len": MAX_LEN,
    "source": SOURCE_MODEL, "all_global_layers": GLOBAL_LAYERS,
    "unmodified_sliding_layers": SLIDING_LAYERS,
    "full_context_control": FULL_WINDOW >= 2*MAX_LEN-1,
    "layer15_historical_audit_cos": {"1024": 0.99972, "512": 0.99490},
    "local": LAYER_RESULTS, "cumulative": CUMULATIVE_RESULTS, "joint": JOINT_REPORT,
    "dev": dev_final, "official_test": test_final, "latency": latency,
    "official_test_used_for_training_or_selection": False,
}
checkpoint = OUTPUT_DIR / f"v29_all_global_W{FULL_WINDOW}.pt"
torch.save({"version": "2.9", "window": FULL_WINDOW,
            "global_layers": GLOBAL_LAYERS,
            "replacement_states": {
                i: state_cpu(student_eval.encoder.layers[i].attn)
                for i in GLOBAL_LAYERS},
            "report": report}, checkpoint)
report_path = OUTPUT_DIR / f"v29_all_global_W{FULL_WINDOW}_report.json"
report_path.write_text(json.dumps(report, indent=2, allow_nan=False))
print("DEV:", json.dumps(dev_final, indent=2))
print("OFFICIAL TEST:", json.dumps(test_final, indent=2))
print("FULL MODEL LATENCY:", json.dumps(latency, indent=2))
print("Checkpoint:", checkpoint, "Report:", report_path)


In [ ]:
#@title 11. Checkpoint reload parity, no copy from Layer15
reloaded, _ = build_checkpoint_model(source_dir)
reloaded.to(device).eval().requires_grad_(False)
reloaded.encoder.config.reference_compile = False
payload = torch.load(checkpoint, map_location="cpu", weights_only=True)
assert payload["global_layers"] == GLOBAL_LAYERS and payload["window"] == FULL_WINDOW
for i in GLOBAL_LAYERS:
    replacement = PDeltaV28FastWindowMemory(
        reloaded.encoder.layers[i].attn, local_window=FULL_WINDOW,
        landmark_stride=LANDMARK_STRIDE, feature_dim=FEATURE_DIM, max_landmarks=32
    ).to(device)
    replacement.load_state_dict(payload["replacement_states"][i], strict=True)
    reloaded.encoder.layers[i].attn = replacement.eval().requires_grad_(False)
sanity = DEV_QUALITY_ITEMS[:min(16, len(DEV_QUALITY_ITEMS))]
original = validate_pair(student_eval, items=sanity)
restored = validate_pair(reloaded, items=sanity)
assert abs(original["agreement"]-restored["agreement"]) < 1e-12
assert abs(original["mean_js"]-restored["mean_js"]) < 1e-5
print("RELOAD PARITY PASS", original, restored)


In [ ]:
#@title 12. Image: converted architecture and compounding error
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 3))
ax.set_xlim(0,len(teacher.encoder.layers)); ax.set_ylim(0,1)
for i in range(len(teacher.encoder.layers)):
    repl = i in GLOBAL_LAYERS
    ax.add_patch(plt.Rectangle((i+.06,.32),.88,.34,
                              color="#2176AE" if repl else "#BAC1C9"))
    ax.text(i+.5,.49,str(i),ha="center",va="center",
            color="white" if repl else "black",fontsize=8)
ax.text(.3,.82,f"Blue = FastWindowMemory W{FULL_WINDOW}, gray = original sliding")
ax.axis("off")
fig.tight_layout()
architecture_image = OUTPUT_DIR / f"architecture_all_global_W{FULL_WINDOW}.png"
fig.savefig(architecture_image,dpi=170,bbox_inches="tight")
plt.show()
fig, ax = plt.subplots(figsize=(9,4))
ax.plot([r["through_layer"] for r in CUMULATIVE_RESULTS],
        [r["e2e"]["agreement"] for r in CUMULATIVE_RESULTS],marker="o")
ax.axhline(.95,color="gray",ls="--",label="95% reference")
ax.set(xlabel="Latest globally replaced layer",ylabel="Decision agreement",
       ylim=(0,1.02),title="Error accumulation after each global layer")
ax.grid(alpha=.25)
ax.legend()
fig.tight_layout()
quality_image=OUTPUT_DIR / f"cumulative_quality_W{FULL_WINDOW}.png"
fig.savefig(quality_image,dpi=170,bbox_inches="tight")
plt.show()
print("Images:",architecture_image,quality_image)


In [ ]:
#@title 13. Export fully standalone model and publish to Hugging Face (optional)
# This is run AFTER conversion, fine-tuning, teacher comparison and reload check.
# It exports all model weights (including the fine-tuned task head), config,
# tokenizer, runtime.py and load_model.py. No TinyCeNN-LM dependency at inference.
from tinycenn_lm.hf_standalone import (
    export_v29_fastwindow, verify_standalone_bundle,
    upload_standalone_to_hub
)
from huggingface_hub import HfApi

bundle_dir = export_v29_fastwindow(
    student_eval, tokenizer,
    output_dir=OUTPUT_DIR / f"hf_standalone_W{FULL_WINDOW}",
    global_layers=GLOBAL_LAYERS,
    window=FULL_WINDOW, max_len=MAX_LEN,
    feature_dim=FEATURE_DIM,
    landmark_stride=LANDMARK_STRIDE,
    max_landmarks=32,
    source_model=SOURCE_MODEL,
    report=report,
)
print("STANDALONE BUNDLE:", bundle_dir)
print("CONTENTS:", sorted(str(p.relative_to(bundle_dir))
                          for p in bundle_dir.rglob("*") if p.is_file()))

if VERIFY_STANDALONE_EXPORT:
    # First verify strict key/shape loading, then CPU inference parity.
    # CUDA FP16 vs CPU FP32 may differ slightly; use conservative tolerances.
    sanity_batch = direct_batch(DEV_QUALITY_ITEMS[:1])
    roundtrip = verify_standalone_bundle(
        bundle_dir, student_eval, sanity_batch, atol=0.02, rtol=0.02
    )
    print("INDEPENDENT STANDALONE LOAD/PARITY PASS:", roundtrip)

if UPLOAD_TO_HF:
    # Token is retrieved from Colab Secrets or an already-authenticated HF session.
    # Never print or store the token in notebook outputs or model files.
    hf_token = None
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
    import os
    hf_token = hf_token or os.getenv("HF_TOKEN") or None
    api = HfApi(token=hf_token)
    try:
        identity = api.whoami()
    except Exception as exc:
        raise RuntimeError(
            "Hugging Face authentication required: add HF_TOKEN with write access "
            "to Colab Secrets, or run huggingface_hub.login() and retry this cell."
        ) from exc
    target_repo = HF_REPO_ID.strip() or (
        f"{identity['name']}/Laya-V29-AllGlobal-W{FULL_WINDOW}-Standalone"
    )
    url = upload_standalone_to_hub(
        bundle_dir, repo_id=target_repo, token=hf_token, private=HF_PRIVATE
    )
    print("HUGGING FACE PUBLISH COMPLETE:", url)
    print("MODEL:", f"https://huggingface.co/{target_repo}")
else:
    print("UPLOAD_TO_HF=False: local bundle saved, no remote upload attempted.")

# Once uploaded, from any fresh Python environment:
# from huggingface_hub import hf_hub_download
# import importlib.util
# path = hf_hub_download("YOUR_USER/YOUR_MODEL", "load_model.py")
# spec = importlib.util.spec_from_file_location("hf_loader", path)
# loader = importlib.util.module_from_spec(spec); spec.loader.exec_module(loader)
# model, tokenizer = loader.load_model("YOUR_USER/YOUR_MODEL", device="cpu")
